In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def show_binary_side_by_side_highlight_new(
    tiles,
    titles=None,
    figsize_per_tile=2.0,
    border=True,
    savepath=None,
    new_color=(0.0, 0.35, 1.0),     # blue for newly-added positions
    old_color=(0.25, 0.25, 0.25),   # gray for previously-added positions
    bg_color=(1.0, 1.0, 1.0)        # white background
):
    """
    Visualize a list/array of 16x16 binary tiles side-by-side,
    coloring newly-added 1s in *this* pass as blue, and previously-added 1s as gray.

    tiles:  list[np.ndarray] or np.ndarray, shape (N,16,16), dtype bool or {0,1}
    titles: optional list[str] of length N
    """

    # Normalize input
    arr = np.asarray(tiles)
    if arr.ndim == 2 and arr.shape == (16, 16):
        arr = arr[None, ...]
    assert arr.ndim == 3 and arr.shape[1:] == (16, 16), f"Expected (N,16,16), got {arr.shape}"

    # Coerce to boolean {False, True}
    if arr.dtype == bool:
        arrb = arr
    else:
        # accept {0,1} or threshold anything else
        unique = np.unique(arr)
        if np.array_equal(unique, [0, 1]) or np.array_equal(unique, [0]) or np.array_equal(unique, [1]):
            arrb = arr.astype(bool)
        else:
            arrb = (arr > 0.5)

    N, H, W = arrb.shape
    if titles is not None:
        assert len(titles) == N, "titles length must match number of tiles"

    # Build colored RGB images per pass
    rgb_tiles = []
    prev_filled = np.zeros((H, W), dtype=bool)
    for k in range(N):
        cur = arrb[k]
        new_mask = cur & (~prev_filled)   # newly-added in this pass
        old_mask = cur & prev_filled      # already-added before this pass

        img = np.empty((H, W, 3), dtype=float)
        img[...] = bg_color
        if old_mask.any():
            img[old_mask] = old_color
        if new_mask.any():
            img[new_mask] = new_color

        rgb_tiles.append(img)
        prev_filled = cur  # update cumulative mask

    # Plot row of images
    fig_w = max(2.0, figsize_per_tile * N)
    fig, axes = plt.subplots(1, N, figsize=(fig_w, figsize_per_tile), squeeze=False)
    axes = axes[0]

    for i, ax in enumerate(axes):
        ax.imshow(rgb_tiles[i], interpolation="nearest", vmin=0.0, vmax=1.0)
        ax.set_xticks([]); ax.set_yticks([])
        if titles is not None:
            ax.set_title(str(titles[i]), fontsize=10, pad=4)
        # optional border
        for spine in ax.spines.values():
            spine.set_visible(border)
            if border:
                spine.set_linewidth(0.8)
                spine.set_color((0.8, 0.8, 0.8))

    # Simple legend (drawn once on the last axes)
    import matplotlib.patches as mpatches
    new_patch = mpatches.Patch(color=new_color, label="new in this pass")
    old_patch = mpatches.Patch(color=old_color, label="previous passes")
    bg_patch  = mpatches.Patch(color=bg_color, label="background")
    # axes[-1].legend(handles=[new_patch, old_patch, bg_patch], loc="upper right", fontsize=8, frameon=True)

    plt.tight_layout()
    if savepath is not None:
        plt.savefig(savepath, dpi=200, bbox_inches="tight")
    plt.show()



def make_interlaced_tiles(
    H=16, W=16, BS=8, N=6,
    start_size=1,            # initial rectangle height/width inside each block
    start_right_first=True   # first expansion is right (width) if True, else bottom (height)
):
    """
    Build a list of N binary tiles (H×W), where each pass doubles the previous
    decoded rectangle *within each BS×BS block*, alternating width then height.
    Area becomes 4x larger every 2 passes (until clamped by BS).

    Returns:
        tiles: list[np.ndarray] of length N, each (H, W) with {0,1}
        sizes: list[tuple[int,int]] per-pass (height, width) inside each block
    """
    assert H % BS == 0 and W % BS == 0, "H and W must be multiples of BS"
    assert 1 <= start_size <= BS, "start_size must be in [1, BS]"

    # per-pass (h, w) inside each block
    sizes = []
    h = w = int(start_size)
    # which axis to double next (True = width, False = height)
    double_width_next = bool(start_right_first)

    for k in range(N):
        if k == 0:
            # first pass: seed rectangle (h,w)
            sizes.append((h, w))
        else:
            if double_width_next:
                w = min(BS, w * 2)  # right-first: double width
            else:
                h = min(BS, h * 2)  # bottom-first: double height
            sizes.append((h, w))
            double_width_next = not double_width_next  # alternate

    tiles = []
    for k in range(N):
        hh, ww = sizes[k]
        t = np.zeros((H, W), dtype=np.uint8)
        # fill the same (top-left anchored) rectangle inside every BS×BS block
        for bi in range(H // BS):
            for bj in range(W // BS):
                y0 = bi * BS
                x0 = bj * BS
                t[y0:y0 + hh, x0:x0 + ww] = 1
        tiles.append(t)

    return tiles, sizes


# --- Example usage (matches your generator) ---
if __name__ == "__main__":
    H, W = 16, 16
    BS = 8
    N = 7   # a 4-pass example
    tiles, _ = make_interlaced_tiles(H=H, W=W, BS=BS, N=N, start_size=1, start_right_first=True)

    show_binary_side_by_side_highlight_new(
        tiles,
        titles=[f"pass {i + 1}" for i in range(N)],
        figsize_per_tile=2.2,
        border=True,
        savepath=None  # e.g., "passes.png"
    )
